# 05i — Model I Training: Uncertainty-Aware / Quantile Models

Predicts the **distribution** of `lap_time_ratio` via quantile regression,
mixture density networks, or deep ensembles.

8 GPU candidates: 2 tree-quantile + 3 DL-quantile + 2 MDN + 1 ensemble.
Quantile Monte Carlo: sample from predicted distribution for principled MC.

CV: ExpandingWindowSplit (2019→2020, ..., 2019-23→2024 test).

## 0. Setup

In [ ]:
import os
from pathlib import Path

if not (Path.cwd() / "pyproject.toml").exists():
    # We're likely in notebooks/ — go up to repo root
    for p in [Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "pyproject.toml").exists():
            os.chdir(p)
            break

print(f"Working directory: {Path.cwd()}")

In [ ]:
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import lightgbm as lgb

from f1_predictor.features.splits import ExpandingWindowSplit, LeaveOneSeasonOut
from f1_predictor.data.storage import (
    load_from_gcs_or_local,
    load_training_parquet,
    save_training_parquet,
    save_model_pickle as gcs_save_model_pickle,
    save_notebook,
    sync_training_from_gcs,
)
from f1_predictor.models.gpu import (
    detect_gpu_backend, get_lightgbm_device, get_torch_device, get_xgboost_device,
)

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

TRAINING_DIR = Path("data/training")
MODEL_DIR = Path("data/raw/model")
TRAINING_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# GPU detection (supports NVIDIA CUDA and AMD ROCm)
GPU_BACKEND, GPU_NAME = detect_gpu_backend()
TORCH_DEVICE = get_torch_device()
print(f"GPU backend: {GPU_BACKEND} ({GPU_NAME})")
print(f"PyTorch device: {TORCH_DEVICE}")

# Deep learning models (PyTorch — works on both CUDA and ROCm via HIP)
DL_AVAILABLE = False
try:
    from f1_predictor.models.architectures import GRU2Layer, FTTransformerWrapper, MLP3Layer
    DL_AVAILABLE = TORCH_DEVICE != "cpu"
    print(f"DL models available: {DL_AVAILABLE}")
except (ImportError, NameError):
    print("DL models not available (torch/rtdl not installed)")

In [ ]:
def save_predictions(model, X, y, id_df, model_type, model_name, split_name):
    """Save prediction parquet locally and to GCS."""
    preds = model.predict(X)
    out = id_df.copy()
    out["y_true"] = y.values
    out["y_pred"] = preds
    fname = f"model_{model_type}_{model_name}_{split_name}.parquet"
    uri = save_training_parquet(out, fname, TRAINING_DIR)
    print(f"  Saved {fname} -> {uri}")
    return preds


def save_model_pkl(model, model_type, model_name):
    """Save model pickle locally and to GCS."""
    fname = f"Model_{model_type}_{model_name}.pkl"
    uri = gcs_save_model_pickle(model, fname, MODEL_DIR)
    print(f"  Saved {fname} -> {uri}")

## 1. Build Training Data

In [ ]:
from f1_predictor.features.simulation_features import (
    build_simulation_training_data,
    SIMULATION_FEATURE_COLS,
)

laps = load_from_gcs_or_local(
    "data/raw/laps/all_laps.parquet",
    Path("data/raw/laps/all_laps.parquet"),
)
races = load_from_gcs_or_local(
    "data/raw/race/all_races.parquet",
    Path("data/raw/race/all_races.parquet"),
)

df = build_simulation_training_data(laps, races)
print(f"Shape: {df.shape}")
print(f"Target stats:\n{df['lap_time_ratio'].describe()}")

In [ ]:
FEATURE_COLS = SIMULATION_FEATURE_COLS
TARGET = "lap_time_ratio"
ID_COLS = ["season", "round", "event_name", "driver_abbrev", "team"]

df = df.dropna(subset=[TARGET]).reset_index(drop=True)

X = df[FEATURE_COLS]
y = df[TARGET]
groups = df["season"].values
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

## 2. CV Splitter

In [ ]:
splitter = ExpandingWindowSplit(
    fold_definitions=[
        ([2019], 2020),
        ([2019, 2020], 2021),
        ([2019, 2020, 2021], 2022),
        ([2019, 2020, 2021, 2022], 2023),
    ],
    test_season=2024,
)
print(f"CV folds: {splitter.get_n_splits()}")
for i, (tr, va) in enumerate(splitter.split(groups)):
    tr_seasons = sorted(set(groups[tr]))
    va_seasons = sorted(set(groups[va]))
    print(f"  Fold {i}: train={tr_seasons}, val={va_seasons}, "
          f"train={len(tr):,}, val={len(va):,}")

## 3. Model Candidates and Helpers

8 uncertainty-aware candidates, all Optuna-tunable.

In [ ]:
from f1_predictor.models.quantile_architectures import (
    LightGBM_Quantile, XGBoost_Quantile,
    MLP_MultiQuantile, GRU_MultiQuantile, FTTransformer_Quantile,
    MDN_MLP, MDN_GRU, DeepEnsemble,
)

DL_SKIP_OPTUNA = set()  # all models tunable
NAN_TOLERANT = {"LightGBM_Quantile", "XGBoost_Quantile"}

MODEL_CLASSES_I = {
    "LightGBM_Quantile": LightGBM_Quantile,
    "XGBoost_Quantile": XGBoost_Quantile,
}
if DL_AVAILABLE:
    MODEL_CLASSES_I.update({
        "MLP_MultiQuantile": MLP_MultiQuantile,
        "GRU_MultiQuantile": GRU_MultiQuantile,
        "FTTransformer_Quantile": FTTransformer_Quantile,
        "MDN_MLP": MDN_MLP,
        "MDN_GRU": MDN_GRU,
        "DeepEnsemble": DeepEnsemble,
    })


def get_candidates_i():
    xgb_device = get_xgboost_device(GPU_BACKEND)
    lgb_device = get_lightgbm_device(GPU_BACKEND)
    candidates = {
        "LightGBM_Quantile": LightGBM_Quantile(
            n_estimators=300, n_jobs=-1, random_state=42, verbose=-1, **lgb_device),
        "XGBoost_Quantile": XGBoost_Quantile(
            n_estimators=300, n_jobs=-1, random_state=42, verbosity=0, **xgb_device),
    }
    if DL_AVAILABLE:
        n_feat = len(FEATURE_COLS)
        candidates["MLP_MultiQuantile"] = MLP_MultiQuantile(input_dim=n_feat)
        candidates["GRU_MultiQuantile"] = GRU_MultiQuantile(input_dim=n_feat)
        candidates["FTTransformer_Quantile"] = FTTransformer_Quantile(input_dim=n_feat)
        candidates["MDN_MLP"] = MDN_MLP(input_dim=n_feat)
        candidates["MDN_GRU"] = MDN_GRU(input_dim=n_feat)
        candidates["DeepEnsemble"] = DeepEnsemble(input_dim=n_feat)
    print(f"Candidates ({len(candidates)}): {list(candidates.keys())}")
    return candidates


def _tree_quantile_space(trial, tree_type):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 1500),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    )
    if tree_type == "xgb":
        xgb_device = get_xgboost_device(GPU_BACKEND)
        params.update(n_jobs=-1, random_state=42, verbosity=0, **xgb_device)
    else:
        lgb_device = get_lightgbm_device(GPU_BACKEND)
        params.update(n_jobs=-1, random_state=42, verbose=-1, **lgb_device)
    return params


def _dl_quantile_space(trial, model_cls):
    params = {"input_dim": len(FEATURE_COLS)}
    if model_cls in (MLP_MultiQuantile, MDN_MLP, DeepEnsemble):
        params["hidden1"] = trial.suggest_categorical("hidden1", [64, 128, 256])
        params["hidden2"] = trial.suggest_categorical("hidden2", [32, 64, 128])
    if model_cls in (GRU_MultiQuantile, MDN_GRU):
        params["hidden_dim"] = trial.suggest_categorical("hidden_dim", [32, 64, 128, 256])
        params["num_layers"] = trial.suggest_int("num_layers", 1, 3)
    if model_cls == FTTransformer_Quantile:
        params["d_token"] = trial.suggest_categorical("d_token", [32, 64, 128])
        params["n_blocks"] = trial.suggest_int("n_blocks", 1, 4)
    params["dropout"] = trial.suggest_float("dropout", 0.05, 0.5)
    params["lr"] = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    params["weight_decay"] = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    if model_cls in (MDN_MLP, MDN_GRU):
        params["n_components"] = trial.suggest_int("n_components", 2, 5)
    if model_cls == DeepEnsemble:
        params["n_members"] = trial.suggest_int("n_members", 3, 7)
    return params


def get_optuna_param_space_i(name, trial):
    if name == "LightGBM_Quantile":
        return _tree_quantile_space(trial, "lgb")
    elif name == "XGBoost_Quantile":
        return _tree_quantile_space(trial, "xgb")
    else:
        return _dl_quantile_space(trial, MODEL_CLASSES_I[name])


def reconstruct_params_i(name, best_params):
    params = dict(best_params)
    if name == "XGBoost_Quantile":
        xgb_device = get_xgboost_device(GPU_BACKEND)
        params.update(n_jobs=-1, random_state=42, verbosity=0, **xgb_device)
    elif name == "LightGBM_Quantile":
        lgb_device = get_lightgbm_device(GPU_BACKEND)
        params.update(n_jobs=-1, random_state=42, verbose=-1, **lgb_device)
    else:
        params.setdefault("input_dim", len(FEATURE_COLS))
    return params

In [ ]:
def cv_evaluate_i(model, X, y, splitter, groups):
    from sklearn.metrics import mean_squared_error, mean_absolute_error
    rmses, maes = [], []
    for tr_idx, va_idx in splitter.split(groups):
        import sklearn.base
        m = sklearn.base.clone(model)
        X_tr = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        y_tr = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        X_va = X.iloc[va_idx] if hasattr(X, "iloc") else X[va_idx]
        y_va = y.iloc[va_idx] if hasattr(y, "iloc") else y[va_idx]
        name_str = type(model).__name__
        if name_str in str(NAN_TOLERANT) or any(n in str(type(model)) for n in ["XGB", "LGB"]):
            m.fit(X_tr, y_tr)
        else:
            X_tr_clean = X_tr.fillna(0) if hasattr(X_tr, "fillna") else np.nan_to_num(X_tr)
            X_va_clean = X_va.fillna(0) if hasattr(X_va, "fillna") else np.nan_to_num(X_va)
            m.fit(X_tr_clean, y_tr)
            X_va = X_va_clean
        # Use median prediction (q50) for RMSE
        preds = m.predict(X_va)
        rmses.append(np.sqrt(mean_squared_error(y_va, preds)))
        maes.append(mean_absolute_error(y_va, preds))
    return {"mean_rmse": np.mean(rmses), "std_rmse": np.std(rmses), "mean_mae": np.mean(maes)}


def screen_models_i(candidates, X, y, splitter, groups):
    rows = []
    for name, model in candidates.items():
        try:
            result = cv_evaluate_i(model, X, y, splitter, groups)
            rows.append({"model": name, **result})
            print(f"  {name}: RMSE={result['mean_rmse']:.4f}")
        except Exception as e:
            print(f"  {name}: FAILED — {e}")
    return pd.DataFrame(rows).sort_values("mean_rmse").reset_index(drop=True)


def run_optuna_round_i(name, X, y, splitter, groups, n_trials):
    def objective(trial):
        params = get_optuna_param_space_i(name, trial)
        model_cls = MODEL_CLASSES_I[name]
        model = model_cls(**params)
        result = cv_evaluate_i(model, X, y, splitter, groups)
        return result["mean_rmse"]
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(
        objective, n_trials=n_trials,
        catch=(Exception,), show_progress_bar=False,
    )
    return study.best_params, study.best_value

## 4. Round 1 — Screen Models

In [ ]:
candidates = get_candidates_i()
r1_results = screen_models_i(candidates, X, y, splitter, groups)
r1_results[["model", "mean_rmse", "std_rmse", "mean_mae"]]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(r1_results["model"], r1_results["mean_rmse"], xerr=r1_results["std_rmse"])
ax.set_xlabel("Mean RMSE (CV)")
ax.set_title("Round 1: Model Screening — Model I (Quantile/Uncertainty)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
top7_names = r1_results["model"].head(min(7, len(r1_results))).tolist()
print(f"Advancing to Round 2: {top7_names}")
eliminated = r1_results["model"].iloc[7:].tolist()
if eliminated:
    print(f"Eliminated: {eliminated}")

## 5. Round 2 — Optuna (top 7, 10 trials each)

In [ ]:
r2_results = []
for name in top7_names:
    print(f"Tuning {name}...")
    best_params, best_rmse = run_optuna_round_i(
        name, X, y, splitter, groups, n_trials=10)
    r2_results.append({"model": name, "best_rmse": best_rmse, "best_params": best_params})
    print(f"  Best RMSE: {best_rmse:.4f}")

r2_df = pd.DataFrame(r2_results).sort_values("best_rmse").reset_index(drop=True)
r2_df[["model", "best_rmse"]]

In [ ]:
top5_names = r2_df["model"].head(5).tolist()
r2_best_params = {row["model"]: row["best_params"] for _, row in r2_df.iterrows()}
print(f"Advancing to Round 3: {top5_names}")

## 6. Round 3 — Final Tuning (top 5, 15 trials each)

In [ ]:
r3_results = []
for name in top5_names:
    print(f"Fine-tuning {name}...")
    best_params, best_rmse = run_optuna_round_i(
        name, X, y, splitter, groups, n_trials=15)
    r3_results.append({"model": name, "best_rmse": best_rmse, "best_params": best_params})
    print(f"  Best RMSE: {best_rmse:.4f}")

r3_df = pd.DataFrame(r3_results).sort_values("best_rmse").reset_index(drop=True)
r3_best_params = {row["model"]: row["best_params"] for _, row in r3_df.iterrows()}
r3_df[["model", "best_rmse"]]

## 7. Test Set Evaluation (Per-Lap)

In [ ]:
train_idx, test_idx = splitter.get_test_split(groups)
X_train_full, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train_full, y_test = y.iloc[train_idx], y.iloc[test_idx]
id_train, id_test = df[ID_COLS].iloc[train_idx], df[ID_COLS].iloc[test_idx]

print(f"Train: {X_train_full.shape}, Test: {X_test.shape}")
print(f"Test season(s): {sorted(df['season'].iloc[test_idx].unique())}")

In [ ]:
final_results = []
for name in top5_names:
    params = reconstruct_params_i(name, r3_best_params[name])
    model_cls = MODEL_CLASSES_I[name]
    model = model_cls(**params)
    model.fit(X_train_full, y_train_full)

    train_preds = model.predict(X_train_full)
    train_rmse = np.sqrt(mean_squared_error(y_train_full, train_preds))

    test_preds = model.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))

    val_rmse = r3_df.loc[r3_df["model"] == name, "best_rmse"].values[0]

    final_results.append({
        "model": name,
        "train_rmse": train_rmse, "val_rmse": val_rmse,
        "test_rmse": test_rmse, "overfit_gap": test_rmse - val_rmse,
    })
    print(f"{name}: train_rmse={train_rmse:.4f}, val_rmse={val_rmse:.4f}, "
          f"test_rmse={test_rmse:.4f}")

final_df = pd.DataFrame(final_results).sort_values("test_rmse").reset_index(drop=True)
final_df

## 8. Quantile Monte Carlo Simulation (2024 Test Races)

In [ ]:
from f1_predictor.simulation.quantile_simulator import QuantileRaceSimulator
from f1_predictor.simulation.defaults import build_circuit_defaults
from f1_predictor.simulation.evaluation import evaluate_simulation, evaluate_monte_carlo_calibration
from f1_predictor.features.race_features import LOCATION_ALIASES

circuit_defaults = build_circuit_defaults(laps)

# Retrain best quantile model on full training set
best_model_name = final_df.iloc[0]["model"]
best_params = reconstruct_params_i(best_model_name, r3_best_params[best_model_name])
best_model = MODEL_CLASSES_I[best_model_name](**best_params)
best_model.fit(X_train_full, y_train_full)

qmc_sim = QuantileRaceSimulator(
    best_model, circuit_defaults, n_simulations=200, seed=42)
print(f"Quantile MC simulator: {best_model_name}, N=200 runs")

In [ ]:
test_races = races[races["season"] == 2024].copy()
test_race_list = test_races.groupby(
    ["season", "round", "event_name"]).first().reset_index()

# Single-run (q50 only)
from f1_predictor.simulation.engine import RaceSimulator
single_sim = RaceSimulator(best_model, circuit_defaults)

sim_results = []
mc_results_all = []

for _, race_row in test_race_list.iterrows():
    event = race_row["event_name"]
    event_norm = LOCATION_ALIASES.get(event, event)

    if event_norm not in circuit_defaults:
        print(f"  Skipping {event} (no circuit data)")
        continue

    race_drivers = test_races[
        (test_races["season"] == race_row["season"])
        & (test_races["round"] == race_row["round"])
    ].copy()

    drivers_input = []
    actual_positions = {}
    for _, drv in race_drivers.iterrows():
        q1 = drv.get("q1_time_sec")
        q2 = drv.get("q2_time_sec")
        q3 = drv.get("q3_time_sec")
        q_times = [t for t in [q1, q2, q3] if pd.notna(t)]
        if not q_times or pd.isna(drv.get("grid_position")):
            continue
        drivers_input.append({
            "driver": drv["driver_abbrev"],
            "grid_position": int(drv["grid_position"]),
            "q1": q1 if pd.notna(q1) else None,
            "q2": q2 if pd.notna(q2) else None,
            "q3": q3 if pd.notna(q3) else None,
            "initial_tyre": "MEDIUM",
        })
        if pd.notna(drv.get("finish_position")):
            actual_positions[drv["driver_abbrev"]] = int(drv["finish_position"])

    if len(drivers_input) < 10:
        continue

    try:
        # Single-run (median) simulation
        result = single_sim.simulate(event_norm, drivers_input)
        for fr in result.final_results:
            if fr["driver"] in actual_positions:
                sim_results.append({
                    "event": event, "driver": fr["driver"],
                    "predicted_pos": fr["position"],
                    "actual_pos": actual_positions[fr["driver"]],
                })

        # Quantile Monte Carlo
        mc_result = qmc_sim.simulate(event_norm, drivers_input)
        for r in mc_result.results:
            if r["driver"] in actual_positions:
                mc_results_all.append({
                    **r, "event": event,
                    "actual_pos": actual_positions[r["driver"]],
                    "predicted_pos": r["position"],
                })
        print(f"  {event}: simulated {len(drivers_input)} drivers")
    except Exception as e:
        print(f"  {event}: failed — {e}")

sim_df = pd.DataFrame(sim_results)
mc_df = pd.DataFrame(mc_results_all)
print(f"\nSingle-run results: {len(sim_df)} driver-race predictions")
print(f"Quantile MC results: {len(mc_df)} driver-race predictions")

In [ ]:
# Single-run metrics
if len(sim_df) > 0:
    single_metrics = evaluate_simulation(sim_df)
    print("=" * 60)
    print("MODEL I — SINGLE-RUN (MEDIAN) SIMULATION (2024)")
    print("=" * 60)
    for k, v in single_metrics.items():
        print(f"  {k:20s}: {v}")

# Quantile MC metrics
if len(mc_df) > 0:
    mc_sim_metrics = evaluate_simulation(mc_df)
    mc_cal_metrics = evaluate_monte_carlo_calibration(mc_df.to_dict("records"))
    print()
    print("=" * 60)
    print("MODEL I — QUANTILE MC SIMULATION (N=200, 2024)")
    print("=" * 60)
    for k, v in mc_sim_metrics.items():
        print(f"  {k:20s}: {v}")
    print()
    print("Calibration:")
    for k, v in mc_cal_metrics.items():
        print(f"  {k:20s}: {v}")

## 9. Save Artifacts

In [ ]:
for name in top5_names:
    params = reconstruct_params_i(name, r3_best_params[name])
    model = MODEL_CLASSES_I[name](**params)
    model.fit(X_train_full, y_train_full)

    save_predictions(model, X_train_full, y_train_full, id_train, "I", name, "Training")
    save_predictions(model, X_test, y_test, id_test, "I", name, "Test")

    oof_preds = np.full(len(X), np.nan)
    for tr_idx, va_idx in splitter.split(groups):
        import sklearn.base
        fold_model = sklearn.base.clone(model)
        fold_model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        oof_preds[va_idx] = fold_model.predict(X.iloc[va_idx])

    val_mask = ~np.isnan(oof_preds)
    val_out = df[ID_COLS].loc[val_mask].copy()
    val_out["y_true"] = y.loc[val_mask].values
    val_out["y_pred"] = oof_preds[val_mask]
    fname = f"model_I_{name}_Validation.parquet"
    uri = save_training_parquet(val_out, fname, TRAINING_DIR)
    print(f"  Saved {fname} -> {uri}")

    save_model_pkl(model, "I", name)

print("\nDone! All Model I artifacts saved.")

## Summary

In [ ]:
print("=" * 60)
print("MODEL I (UNCERTAINTY / QUANTILE) TRAINING COMPLETE")
print("=" * 60)
print(f"\nPer-lap evaluation (top 5, sorted by test RMSE):")
for _, row in final_df.iterrows():
    print(f"  {row['model']:30s}  test_rmse={row['test_rmse']:.6f}  gap={row['overfit_gap']:.6f}")

if len(sim_df) > 0:
    print(f"\nSingle-run (median) simulation (2024):")
    print(f"  Position RMSE: {single_metrics['position_rmse']:.4f}")
    print(f"  Spearman:      {single_metrics['spearman_mean']:.4f}")
    print(f"  Within-3:      {single_metrics['within_3']:.1f}%")

if len(mc_df) > 0:
    print(f"\nQuantile MC simulation (N=200, 2024):")
    print(f"  Position RMSE: {mc_sim_metrics['position_rmse']:.4f}")
    print(f"  Spearman:      {mc_sim_metrics['spearman_mean']:.4f}")
    print(f"  Within-3:      {mc_sim_metrics['within_3']:.1f}%")
    print(f"  Coverage@80%:  {mc_cal_metrics['coverage_80']:.1f}%")

print(f"\nArtifacts saved to:")
print(f"  Predictions: {TRAINING_DIR}")
print(f"  Models: {MODEL_DIR}")